# OpenET Data in Google Earth Engine (GEE) - Python Examples

This notebook contains examples of using the Google Earth Engine Python API to compute spatial reductions of OpenET monthly data.

### Import the Earth Engine API module

In [ ]:
import ee

import pprint

### Initialize Earth Engine using your Cloud Project ID 

You will have to have a Google Cloud Project ID with Earth Engine enabled to run these examples

In [ ]:
ee.Initialize(project='YOUR_PROJECT_ID')

---

### JavaScript vs Python

These are a few of the main differences between the Code Editor and Python:
* You can't show layers on a map without using additional/external modules (geemap)
* You don't need "var " in front of variable declarations
* You don't need to terminate lines with semicolons
* Comments are made with a "#" instead of "//"
* Print statements need a .getInfo() call
* Booleans are mixed case in Python (True or False)
* Other miscellaneous formatting issues


---

## Example 1 - Compute mean annual ET for the "Modesto" Basin

This example will mimic the example we just did in the Code Editor


In [ ]:
# Load the OpenET California/CIMIS Monthly Ensemble collection
ensemble_coll = (
    ee.ImageCollection('projects/openet/assets/ensemble/california/cimis/monthly/v2_1')
    .filterDate('2025-01-01', '2026-01-01')
    .select(['et_ensemble_mad'])
)
# Sum all monthly images in the year
ensemble_img = ensemble_coll.sum()

# Load the ancillary assets
basins_coll = ee.FeatureCollection('projects/ee-cmorton/assets/ca_gw_basins')
mask_img = ee.Image('projects/csumb-et-tools/assets/ca2024_urbanmask')

# Pick a single groundwater basin for testing
basin_ftr = ee.Feature(
    basins_coll
    .filterMetadata('Basin_Su_1', 'equals', 'SAN JOAQUIN VALLEY - MODESTO')
    .first()
)

# Apply the mask to the ET image
ensemble_img = ensemble_img.updateMask(mask_img)

# Compute the mean ET for the basin
mean_et = (
    ensemble_img
    .reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=basin_ftr.geometry(),
        scale=30,
        bestEffort=False,
    )
    .getInfo()
)
print(mean_et)

# Convert the ET value from mm to feet
print(f'\nMean ET: {mean_et["et_ensemble_mad"] / 304.8:0.2f} ft')


---

## Example 2 - Compute Multiple Statistics

Same example as above, except compute both the mean and median in one call by combining reducers

Note, some comments were removed and some lines were combined to simplify reading, and loading ancillary assets was moved to the top

In [ ]:
# Load the ancillary assets
basins_coll = ee.FeatureCollection('projects/ee-cmorton/assets/ca_gw_basins')
mask_img = ee.Image('projects/csumb-et-tools/assets/ca2024_urbanmask')
basin_ftr = ee.Feature(
    basins_coll
    .filterMetadata('Basin_Su_1', 'equals', 'SAN JOAQUIN VALLEY - MODESTO')
    .first()
)

# Compute the sum of the monthly images for the time range
ensemble_img = (
    ee.ImageCollection('projects/openet/assets/ensemble/california/cimis/monthly/v2_1')
    .filterDate('2025-01-01', '2026-01-01')
    .select(['et_ensemble_mad'])
    .sum()
)

# Compute the mean ET for the basin
mean_et = (
    ensemble_img
    .updateMask(mask_img)
    .reduceRegion(
        reducer=ee.Reducer.mean()
            .combine(ee.Reducer.median(), sharedInputs=True),
        geometry=basin_ftr.geometry(),
        scale=30,
        bestEffort=False,
    )
    .getInfo()
)
print(mean_et)

# Convert the ET value from mm to feet
print(f'\nMean ET:   {mean_et["et_ensemble_mad_mean"] / 304.8:0.2f} ft')
print(f'Median ET: {mean_et["et_ensemble_mad_median"] / 304.8:0.2f} ft')


---

## Example 3 - Iterate by year

Process a range of years, in this case 2020-2025.

The ancillary assets only need to be initialized once, so they are called at the beginning (before the year loop).

The two steps to comput the sum image and then compute the mean ET for the basin were combined into a single call to demonstrate how GEE commands can be chained together.

In [ ]:
basins_coll = ee.FeatureCollection('projects/ee-cmorton/assets/ca_gw_basins')
mask_img = ee.Image('projects/csumb-et-tools/assets/ca2024_urbanmask')
basin_ftr = ee.Feature(basins_coll.filterMetadata('Basin_Su_1', 'equals', 'SAN JOAQUIN VALLEY - MODESTO').first())

for year in range(2020, 2026):
    # Compute the mean ET for the basin
    mean_et = (
        ee.ImageCollection('projects/openet/assets/ensemble/california/cimis/monthly/v2_1')
        .filterDate(f'{year}-01-01', f'{year+1}-01-01')
        .select(['et_ensemble_mad'])
        .sum()
        .updateMask(mask_img)
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=basin_ftr.geometry(),
            scale=30,
            bestEffort=False,
        )
        .getInfo()
    )
    # print(mean_et)
    
    # Convert the ET value from mm to feet
    print(f'{year} Mean ET: {mean_et["et_ensemble_mad"] / 304.8:0.2f} ft')


---

## Example 4 - Iterate by basin

Iterate through each of the San Joaqin Valley basins

In [ ]:
basins_coll = ee.FeatureCollection('projects/ee-cmorton/assets/ca_gw_basins')
mask_img = ee.Image('projects/csumb-et-tools/assets/ca2024_urbanmask')

# Build a list of "Basin_Su_1" ID values that have a "Basin_Name" of "SAN JOAQUIN VALLEY"
basin_id_list = (
    basins_coll
    .filterMetadata('Basin_Name', 'equals', 'SAN JOAQUIN VALLEY')
    .aggregate_array('Basin_Su_1')
    .getInfo()
)

for basin_id in basin_id_list:
    # Select the target basin feature
    basin_ftr = ee.Feature(basins_coll.filterMetadata('Basin_Su_1', 'equals', basin_id).first())
    
    # Compute the mean ET for the basin
    mean_et = (
        ee.ImageCollection('projects/openet/assets/ensemble/california/cimis/monthly/v2_1')
        .filterDate(f'{year}-01-01', f'{year+1}-01-01')
        .select(['et_ensemble_mad'])
        .sum()
        .updateMask(mask_img)
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=basin_ftr.geometry(),
            scale=30,
            bestEffort=False,
        )
        .getInfo()
    )
    # print(mean_et)
    
    # Convert the ET value from mm to feet
    print(f'{basin_id:42s} Mean ET: {mean_et["et_ensemble_mad"] / 304.8:0.2f} ft')